In [3]:
!pip install -q geopandas pyogrio rasterio scikit-learn joblib

In [4]:
# ============================================================
# PATCH-BASED CALIBRATED RF ENSEMBLE UNCERTAINTY
# MAINLAND PORTUGAL
#
# ============================================================
#
# NEW WORKFLOW
#
# National 33-band raster
#          ↓
# Divide into LARGE 4096 × 4096 patches
#          ↓
# Within each patch:
#     process 512 × 512 prediction chunks
#          ↓
# Accumulate uncertainty in RAM for the large patch
#          ↓
# Write completed 4096 × 4096 patch ONCE
#          ↓
# Save patch to Google Drive
#          ↓
# Mark patch complete
#          ↓
# Continue to next patch
#
# After all patches:
#
# Patch 001
# Patch 002
# Patch 003
# ...
#          ↓
# Mosaic
#          ↓
# ONE final national uncertainty GeoTIFF
#
#
# RESUME:
#
# If Colab stops:
#
# - completed patches are NOT recalculated
# - empty patches are NOT recalculated
# - corrupt/incomplete patch is recalculated
# - processing resumes from first unfinished patch
#
# Mosaic also has independent resume/checkpoint.
#
#
# SCIENTIFIC METHOD IS UNCHANGED:
#
# B = 50 RF models
# n_estimators = 1000/model
#
# mu(x) = ensemble mean
#
# s_RF(x) =
# sqrt[
#   1/(B-1) *
#   sum(yhat_b(x) - mu(x))²
# ]
#
# z_i =
# (y_i - mu_i) / s_RF,i
#
# k =
# sqrt(mean(z_i²))
#
# U_RF(x) =
# k * s_RF(x)
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import gc
import json
import math
import os
import shutil
import time
from pathlib import Path

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
import rasterio

from rasterio.windows import Window
from rasterio.windows import transform as window_transform
from rasterio.errors import RasterioIOError

from sklearn import config_context
from sklearn.ensemble import RandomForestRegressor


# ============================================================
# 2. INPUT PATHS
# ============================================================

DEVELOPMENT_GPKG = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/"
    "1_Feature_Selection/Split_features/"
    "LUCAS_samples_GSE_train_test.gpkg"
)


VALIDATION_GPKG = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/"
    "1_Feature_Selection/Split_features/"
    "LUCAS_samples_GSE_validation.gpkg"
)


# ONE physical national 33-band GeoTIFF
INPUT_STACKED_RASTER = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/"
    "SOC_GEE_exports/GSE_2018_Imagery_mosaic/"
    "GSE_2018_selected_33_bands_mainland_Portugal.tif"
)


# ============================================================
# 3. OUTPUT PATHS
# ============================================================

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/1.SOC_Estimation/"
    "1.Final_SOC_estimation/"
    "4.Uncertainty_RF_calibrated_ensemble"
)


MODEL_DIR = (
    OUTPUT_ROOT /
    "ensemble_models"
)


# ------------------------------------------------------------
# LARGE uncertainty patches
# ------------------------------------------------------------

PATCH_DIR = (
    OUTPUT_ROOT /
    "uncertainty_large_patches"
)


# ------------------------------------------------------------
# Temporary local patch
#
# Each 4096×4096 patch is first written/compressed locally,
# then copied once to Google Drive.
# ------------------------------------------------------------

LOCAL_PATCH_FILE = Path(
    "/content/"
    "current_uncertainty_patch.tif"
)


# ------------------------------------------------------------
# Final national mosaic
# ------------------------------------------------------------

FINAL_UNCERTAINTY_RASTER = (
    OUTPUT_ROOT /
    "SOC_RF_calibrated_uncertainty_Mainland_Portugal.tif"
)


# Mosaic is first written here.
# If mosaic process stops, this file is reused.
MOSAIC_PARTIAL_RASTER = (
    OUTPUT_ROOT /
    "SOC_RF_calibrated_uncertainty_Mainland_Portugal.partial.tif"
)


# ============================================================
# 4. SMALL OUTPUT FILES
# ============================================================

CALIBRATION_POINTS_CSV = (
    OUTPUT_ROOT /
    "calibration_point_diagnostics.csv"
)


CALIBRATION_SUMMARY_CSV = (
    OUTPUT_ROOT /
    "calibration_summary.csv"
)


PATCH_MANIFEST_CSV = (
    OUTPUT_ROOT /
    "uncertainty_patch_manifest.csv"
)


RUN_CONFIG_JSON = (
    OUTPUT_ROOT /
    "uncertainty_patch_run_config.json"
)


MOSAIC_CHECKPOINT_FILE = (
    OUTPUT_ROOT /
    "uncertainty_mosaic_checkpoint.json"
)


# ============================================================
# 5. EXACT PREDICTORS
# ============================================================

TARGET_COLUMN = "OC"

TARGET_CRS = "EPSG:3763"


SELECTED_FEATURES = [

    "A01",
    "A05",
    "A06",
    "A07",
    "A08",
    "A10",
    "A13",
    "A14",
    "A16",
    "A19",
    "A21",
    "A23",
    "A26",
    "A27",
    "A30",
    "A31",
    "A33",
    "A34",
    "A35",
    "A37",
    "A38",
    "A39",
    "A40",
    "A41",
    "A45",
    "A48",
    "A49",
    "A50",
    "A51",
    "A53",
    "A54",
    "A56",
    "A62",
]


N_FEATURES = len(
    SELECTED_FEATURES
)


if N_FEATURES != 33:

    raise ValueError(
        f"Expected 33 predictors, "
        f"found {N_FEATURES}."
    )


# ============================================================
# 6. RANDOM FOREST SETTINGS
# ============================================================

# Scientific methodology unchanged

B = 50

BASE_RANDOM_SEED = 1000


RF_FIXED_PARAMETERS = {

    "bootstrap": True,

    "max_depth": 20,

    "max_features": "sqrt",

    "min_samples_leaf": 1,

    "n_estimators": 1000,
}


RF_N_JOBS = -1


# ============================================================
# 7. PATCH PROCESSING SETTINGS
# ============================================================

# ------------------------------------------------------------
# LARGE OUTPUT PATCH
#
# 4096 × 4096 Float32:
#
# ~67 MB uncompressed output array.
#
# This is normally safe in Colab while still greatly
# reducing output write operations.
# ------------------------------------------------------------

PATCH_SIZE = 4096


# ------------------------------------------------------------
# INTERNAL RF PREDICTION CHUNK
#
# We DO NOT attempt to load 33 × 4096 × 4096 predictors.
#
# Each large patch is internally processed in 512×512 pieces.
# ------------------------------------------------------------

PREDICTION_WINDOW_SIZE = 512


# 512×512 = maximum 262,144 pixels
PREDICTION_BATCH_SIZE = 300_000


OUTPUT_NODATA = -9999.0


MIN_RAW_SPREAD = 1e-6


# ============================================================
# 8. PATCH OUTPUT SETTINGS
# ============================================================

PATCH_COMPRESSION = "DEFLATE"

PATCH_ZLEVEL = 1


# ============================================================
# 9. MOSAIC OUTPUT SETTINGS
# ============================================================

MOSAIC_COMPRESSION = "DEFLATE"

MOSAIC_ZLEVEL = 1


# Save mosaic progress every N patches.
MOSAIC_CHECKPOINT_EVERY = 5


# ============================================================
# 10. RESTART SETTINGS
# ============================================================

RESUME = True


# ------------------------------------------------------------
# Set True ONLY if you want to delete all existing uncertainty
# patches and calculate everything again.
# ------------------------------------------------------------

OVERWRITE_PATCHES = False


# ------------------------------------------------------------
# Set True ONLY if you want to rebuild the national mosaic.
#
# Existing patches are NOT deleted.
# ------------------------------------------------------------

FORCE_REBUILD_MOSAIC = False


OVERWRITE_ENSEMBLE_MODELS = False


# ============================================================
# 11. RASTER READ SETTINGS
# ============================================================

RASTER_READ_RETRIES = 3

RASTER_READ_RETRY_SECONDS = 3


# ============================================================
# 12. PERFORMANCE
# ============================================================

GC_EVERY_INTERNAL_WINDOWS = 50


# ============================================================
# 13. GENERAL HELPERS
# ============================================================

def banner(message):

    print(
        "\n"
        + "=" * 100
    )

    print(message)

    print(
        "=" * 100,
        flush=True
    )


def step(message):

    print(
        f"[{time.strftime('%H:%M:%S')}] "
        f"{message}",
        flush=True
    )


def human_size(num_bytes):

    value = float(
        num_bytes
    )


    for unit in [
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ]:

        if value < 1024:

            return (
                f"{value:.2f} {unit}"
            )


        value /= 1024


    return (
        f"{value:.2f} PB"
    )


# ============================================================
# 14. CREATE OUTPUT DIRECTORIES
# ============================================================

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


PATCH_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 15. INPUT CHECK
# ============================================================

def check_inputs():

    required = [

        DEVELOPMENT_GPKG,

        VALIDATION_GPKG,

        INPUT_STACKED_RASTER,
    ]


    for path in required:

        if not path.exists():

            raise FileNotFoundError(
                f"\nFile not found:\n"
                f"{path}"
            )


# ============================================================
# 16. COLUMN STANDARDIZATION
# ============================================================

def standardize_columns(
    dataframe,
    required_columns,
    dataset_name,
):

    lookup = {

        str(column)
        .strip()
        .casefold(): column

        for column
        in dataframe.columns
    }


    rename_map = {}

    missing = []


    for required in required_columns:

        key = required.casefold()


        if key not in lookup:

            missing.append(
                required
            )

            continue


        actual = lookup[
            key
        ]


        if actual != required:

            rename_map[
                actual
            ] = required


    if missing:

        raise ValueError(

            f"{dataset_name} missing variables:\n"

            + "\n".join(
                f" - {name}"
                for name
                in missing
            )
        )


    if rename_map:

        dataframe = dataframe.rename(
            columns=rename_map
        )


    return dataframe


# ============================================================
# 17. LOAD POINT DATA
# ============================================================

def load_point_dataset(
    path,
    dataset_name,
):

    step(
        f"Loading {dataset_name}"
    )


    gdf = gpd.read_file(
        path
    )


    if gdf.empty:

        raise ValueError(
            f"{dataset_name} is empty."
        )


    if gdf.crs is None:

        raise ValueError(
            f"{dataset_name} has no CRS."
        )


    valid_geometry = (

        gdf.geometry.notna()

        & (~gdf.geometry.is_empty)

        & gdf.geometry.geom_type.eq(
            "Point"
        )
    )


    gdf = gdf.loc[
        valid_geometry
    ].copy()


    required_variables = [

        TARGET_COLUMN,

        *SELECTED_FEATURES,
    ]


    gdf = standardize_columns(

        gdf,

        required_variables,

        dataset_name
    )


    if (
        gdf.crs.to_string()
        != TARGET_CRS
    ):

        gdf = gdf.to_crs(
            TARGET_CRS
        )


    for variable in required_variables:

        gdf[
            variable
        ] = pd.to_numeric(

            gdf[
                variable
            ],

            errors="coerce"
        )


    gdf[
        required_variables
    ] = (

        gdf[
            required_variables
        ]

        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )


    before = len(
        gdf
    )


    gdf = gdf.dropna(

        subset=required_variables

    ).copy()


    removed = (
        before
        - len(gdf)
    )


    if removed:

        step(
            f"{dataset_name}: "
            f"removed {removed} incomplete rows."
        )


    return gdf.reset_index(
        drop=True
    )


# ============================================================
# 18. RANDOM FOREST
# ============================================================

def make_rf(seed):

    return RandomForestRegressor(

        **RF_FIXED_PARAMETERS,

        random_state=int(
            seed
        ),

        n_jobs=RF_N_JOBS
    )


def model_path_for_seed(seed):

    return (

        MODEL_DIR /

        f"RF_ensemble_seed_{seed}.joblib"
    )


# ============================================================
# 19. TRAIN / LOAD ENSEMBLE
# ============================================================

def train_or_load_ensemble(
    X_development,
    y_development,
):

    models = []


    seeds = [

        BASE_RANDOM_SEED + i

        for i in range(B)
    ]


    banner(
        "TRAIN / LOAD B=50 RF MODELS"
    )


    for number, seed in enumerate(
        seeds,
        start=1
    ):

        path = model_path_for_seed(
            seed
        )


        # ====================================================
        # LOAD
        # ====================================================

        if (

            path.exists()

            and not OVERWRITE_ENSEMBLE_MODELS

        ):

            saved = joblib.load(
                path
            )


            if (

                not isinstance(
                    saved,
                    dict
                )

                or "model" not in saved

            ):

                raise ValueError(
                    f"Unexpected model file:\n"
                    f"{path}"
                )


            stored_features = saved.get(

                "selected_features",

                []
            )


            if list(
                stored_features
            ) != SELECTED_FEATURES:

                raise ValueError(

                    f"\nFeature sequence mismatch:\n"
                    f"{path}\n\n"

                    f"Stored:\n"
                    f"{stored_features}\n\n"

                    f"Expected:\n"
                    f"{SELECTED_FEATURES}"
                )


            stored_parameters = saved.get(
                "rf_parameters"
            )


            if (
                stored_parameters is not None

                and stored_parameters
                != RF_FIXED_PARAMETERS
            ):

                raise ValueError(

                    f"\nRF parameter mismatch:\n"
                    f"{path}\n\n"

                    f"Stored:\n"
                    f"{stored_parameters}\n\n"

                    f"Expected:\n"
                    f"{RF_FIXED_PARAMETERS}"
                )


            model = saved[
                "model"
            ]


            if hasattr(
                model,
                "n_jobs"
            ):

                model.n_jobs = (
                    RF_N_JOBS
                )


            step(
                f"Loaded RF "
                f"{number:02d}/{B}"
            )


        # ====================================================
        # TRAIN
        # ====================================================

        else:

            step(
                f"Training RF "
                f"{number:02d}/{B}"
            )


            model = make_rf(
                seed
            )


            model.fit(

                X_development,

                y_development
            )


            joblib.dump(

                {

                    "model":
                        model,

                    "seed":
                        seed,

                    "selected_features":
                        SELECTED_FEATURES.copy(),

                    "target_column":
                        TARGET_COLUMN,

                    "rf_parameters":
                        RF_FIXED_PARAMETERS,
                },

                path,

                compress=3
            )


        models.append(
            model
        )


    return models


# ============================================================
# 20. ENSEMBLE MEAN + STANDARD DEVIATION
# ============================================================

def predict_with_ensemble(
    models,
    X
):

    X = np.ascontiguousarray(

        X,

        dtype=np.float32
    )


    if (
        X.ndim != 2

        or X.shape[1]
        != N_FEATURES
    ):

        raise ValueError(
            f"Invalid predictor matrix: "
            f"{X.shape}"
        )


    n_samples = (
        X.shape[0]
    )


    running_mean = np.zeros(

        n_samples,

        dtype=np.float64
    )


    running_m2 = np.zeros(

        n_samples,

        dtype=np.float64
    )


    count = 0


    with config_context(
        assume_finite=True
    ):

        for model in models:

            prediction = (

                model.predict(
                    X
                )

                .astype(
                    np.float64,
                    copy=False
                )
            )


            count += 1


            delta = (

                prediction
                - running_mean
            )


            running_mean += (

                delta
                / count
            )


            running_m2 += (

                delta

                * (
                    prediction
                    - running_mean
                )
            )


    raw_spread = np.sqrt(

        running_m2
        / (count - 1)
    )


    return (
        running_mean,
        raw_spread
    )


# ============================================================
# 21. CALIBRATION FACTOR
# ============================================================

def derive_scaling_factor(
    models,
    validation
):

    banner(
        "CALCULATE GLOBAL CALIBRATION FACTOR k"
    )


    X_validation = (

        validation[
            SELECTED_FEATURES
        ]

        .to_numpy(
            dtype=np.float32
        )
    )


    y_validation = (

        validation[
            TARGET_COLUMN
        ]

        .to_numpy(
            dtype=np.float64
        )
    )


    ensemble_mean, raw_spread = (
        predict_with_ensemble(

            models,

            X_validation
        )
    )


    valid = (

        np.isfinite(
            y_validation
        )

        & np.isfinite(
            ensemble_mean
        )

        & np.isfinite(
            raw_spread
        )

        & (
            raw_spread
            > MIN_RAW_SPREAD
        )
    )


    observed = (
        y_validation[
            valid
        ]
    )


    predicted = (
        ensemble_mean[
            valid
        ]
    )


    spread = (
        raw_spread[
            valid
        ]
    )


    residual = (

        observed
        - predicted
    )


    z = (

        residual
        / spread
    )


    k = float(

        np.sqrt(

            np.mean(
                np.square(
                    z
                )
            )
        )
    )


    calibrated = (

        k
        * spread
    )


    calibrated_z = (

        residual
        / calibrated
    )


    # ========================================================
    # SAVE POINT DIAGNOSTICS
    # ========================================================

    pd.DataFrame({

        "Observed_SOC":
            observed,

        "Ensemble_mean_SOC":
            predicted,

        "Residual":
            residual,

        "Absolute_residual":
            np.abs(
                residual
            ),

        "Raw_RF_spread":
            spread,

        "Standardized_residual_z":
            z,

        "Scaling_factor_k":
            k,

        "Calibrated_uncertainty":
            calibrated,

        "Calibrated_z":
            calibrated_z,

    }).to_csv(

        CALIBRATION_POINTS_CSV,

        index=False
    )


    pd.DataFrame([{

        "n_validation_points":
            int(
                valid.sum()
            ),

        "B_RF_models":
            B,

        "trees_per_RF":
            RF_FIXED_PARAMETERS[
                "n_estimators"
            ],

        "n_predictors":
            N_FEATURES,

        "scaling_factor_k":
            k,

        "raw_spread_mean":
            float(
                np.mean(
                    spread
                )
            ),

        "raw_spread_min":
            float(
                np.min(
                    spread
                )
            ),

        "raw_spread_max":
            float(
                np.max(
                    spread
                )
            ),

        "calibrated_uncertainty_mean":
            float(
                np.mean(
                    calibrated
                )
            ),

        "calibrated_uncertainty_min":
            float(
                np.min(
                    calibrated
                )
            ),

        "calibrated_uncertainty_max":
            float(
                np.max(
                    calibrated
                )
            ),

    }]).to_csv(

        CALIBRATION_SUMMARY_CSV,

        index=False
    )


    print(
        f"\nValidation points: "
        f"{valid.sum()}"
    )


    print(
        f"k = "
        f"{k:.8f}"
    )


    return k


# ============================================================
# 22. BAND ALIGNMENT
# ============================================================

def resolve_raster_band_indices(
    src
):

    if src.count != N_FEATURES:

        raise ValueError(
            f"Raster has "
            f"{src.count} bands; "
            "expected 33."
        )


    descriptions = [

        (
            str(x).strip()
            if x is not None
            else None
        )

        for x
        in src.descriptions
    ]


    if all(
        x is not None
        for x
        in descriptions
    ):

        if (
            set(descriptions)
            != set(SELECTED_FEATURES)
        ):

            raise ValueError(
                "Raster predictor names "
                "do not match selected predictors."
            )


        lookup = {

            name: index

            for index, name
            in enumerate(
                descriptions,
                start=1
            )
        }


        return [

            lookup[
                feature
            ]

            for feature
            in SELECTED_FEATURES
        ]


    print(
        "\nWARNING:"
        "\nUsing positional raster bands 1–33."
    )


    return list(
        range(
            1,
            34
        )
    )


# ============================================================
# 23. LARGE PATCH SPECIFICATIONS
# ============================================================

def build_patch_specs(
    width,
    height
):

    specs = []


    patch_rows = math.ceil(
        height / PATCH_SIZE
    )


    patch_cols = math.ceil(
        width / PATCH_SIZE
    )


    patch_index = 0


    for patch_row in range(
        patch_rows
    ):

        row_off = (
            patch_row
            * PATCH_SIZE
        )


        patch_height = min(

            PATCH_SIZE,

            height - row_off
        )


        for patch_col in range(
            patch_cols
        ):

            col_off = (
                patch_col
                * PATCH_SIZE
            )


            patch_width = min(

                PATCH_SIZE,

                width - col_off
            )


            patch_index += 1


            patch_id = (

                f"r{patch_row:03d}_"
                f"c{patch_col:03d}"
            )


            window = Window(

                col_off=col_off,

                row_off=row_off,

                width=patch_width,

                height=patch_height
            )


            specs.append({

                "index":
                    patch_index,

                "patch_id":
                    patch_id,

                "patch_row":
                    patch_row,

                "patch_col":
                    patch_col,

                "window":
                    window,

                "width":
                    patch_width,

                "height":
                    patch_height,
            })


    return specs


# ============================================================
# 24. PATCH PATHS
# ============================================================

def patch_tif_path(
    patch_id
):

    return (

        PATCH_DIR /

        f"SOC_RF_uncertainty_{patch_id}.tif"
    )


def patch_empty_marker(
    patch_id
):

    return (

        PATCH_DIR /

        f"SOC_RF_uncertainty_{patch_id}.empty.json"
    )


def patch_partial_path(
    patch_id
):

    return Path(

        str(
            patch_tif_path(
                patch_id
            )
        )

        + ".partial"
    )


# ============================================================
# 25. GENERATE INTERNAL WINDOWS INSIDE LARGE PATCH
# ============================================================

def generate_internal_windows(
    patch_window
):

    patch_row0 = int(
        patch_window.row_off
    )


    patch_col0 = int(
        patch_window.col_off
    )


    patch_height = int(
        patch_window.height
    )


    patch_width = int(
        patch_window.width
    )


    for local_row in range(

        0,

        patch_height,

        PREDICTION_WINDOW_SIZE
    ):

        sub_height = min(

            PREDICTION_WINDOW_SIZE,

            patch_height
            - local_row
        )


        for local_col in range(

            0,

            patch_width,

            PREDICTION_WINDOW_SIZE
        ):

            sub_width = min(

                PREDICTION_WINDOW_SIZE,

                patch_width
                - local_col
            )


            national_window = Window(

                col_off=(
                    patch_col0
                    + local_col
                ),

                row_off=(
                    patch_row0
                    + local_row
                ),

                width=sub_width,

                height=sub_height
            )


            yield (

                national_window,

                local_row,

                local_col,

                sub_height,

                sub_width
            )


# ============================================================
# 26. QUICK DATA TEST
# ============================================================

def window_has_possible_data(
    src,
    first_band,
    window
):

    data = src.read(

        first_band,

        window=window,

        out_dtype="float32",

        masked=False
    )


    valid = np.isfinite(
        data
    )


    nodata = (
        src.nodatavals[
            first_band - 1
        ]
    )


    if (

        nodata is not None

        and np.isfinite(
            nodata
        )

    ):

        valid &= (
            data != nodata
        )


    elif nodata is None:

        mask = src.read_masks(

            first_band,

            window=window
        )


        valid &= (
            mask > 0
        )


        del mask


    result = bool(
        np.any(
            valid
        )
    )


    del data
    del valid


    return result


# ============================================================
# 27. READ PREDICTOR WINDOW
# ============================================================

def read_predictor_window(
    src,
    band_indices,
    window
):

    last_error = None


    for attempt in range(
        1,
        RASTER_READ_RETRIES + 1
    ):

        try:

            data = src.read(

                indexes=band_indices,

                window=window,

                out_dtype="float32",

                masked=False
            )


            valid = np.all(

                np.isfinite(
                    data
                ),

                axis=0
            )


            all_have_nodata = True


            for array_position, raster_band in enumerate(
                band_indices
            ):

                nodata = (

                    src.nodatavals[
                        raster_band - 1
                    ]
                )


                if nodata is None:

                    all_have_nodata = False

                    continue


                if np.isnan(
                    nodata
                ):

                    continue


                valid &= (

                    data[
                        array_position
                    ]

                    != nodata
                )


            if not all_have_nodata:

                masks = src.read_masks(

                    indexes=band_indices,

                    window=window
                )


                valid &= np.all(

                    masks > 0,

                    axis=0
                )


                del masks


            return (
                data,
                valid
            )


        except RasterioIOError as error:

            last_error = error


            print(
                f"\nRaster read failed "
                f"{attempt}/"
                f"{RASTER_READ_RETRIES}"
            )


            if (
                attempt
                < RASTER_READ_RETRIES
            ):

                time.sleep(
                    RASTER_READ_RETRY_SECONDS
                )


    raise RuntimeError(

        f"\nFailed to read:\n"
        f"{window}\n\n"

        f"{last_error}"
    )


# ============================================================
# 28. PREDICT SPREAD IN BATCHES
# ============================================================

def predict_spread_in_batches(
    models,
    X
):

    n_samples = (
        X.shape[0]
    )


    spread = np.empty(

        n_samples,

        dtype=np.float64
    )


    for start in range(

        0,

        n_samples,

        PREDICTION_BATCH_SIZE
    ):

        stop = min(

            start
            + PREDICTION_BATCH_SIZE,

            n_samples
        )


        _, batch_spread = (
            predict_with_ensemble(

                models,

                X[
                    start:stop
                ]
            )
        )


        spread[
            start:stop
        ] = batch_spread


    return spread


# ============================================================
# 29. VALIDATE EXISTING PATCH
# ============================================================

def patch_is_valid(
    patch_path,
    expected_window,
    source
):

    if not patch_path.exists():

        return False


    try:

        expected_transform = (
            window_transform(

                expected_window,

                source.transform
            )
        )


        with rasterio.open(
            patch_path
        ) as patch:


            checks = [

                patch.count == 1,

                patch.width
                == int(
                    expected_window.width
                ),

                patch.height
                == int(
                    expected_window.height
                ),

                patch.crs
                == source.crs,

                patch.transform
                == expected_transform,

                patch.dtypes[0]
                == "float32",
            ]


            return all(
                checks
            )


    except Exception:

        return False


# ============================================================
# 30. PATCH STATUS
# ============================================================

def get_patch_status(
    spec,
    source
):

    patch_id = (
        spec[
            "patch_id"
        ]
    )


    tif = patch_tif_path(
        patch_id
    )


    empty = patch_empty_marker(
        patch_id
    )


    partial = patch_partial_path(
        patch_id
    )


    # Remove stale partial output.
    if partial.exists():

        partial.unlink(
            missing_ok=True
        )


    if empty.exists():

        return "EMPTY"


    if tif.exists():

        if patch_is_valid(

            tif,

            spec[
                "window"
            ],

            source

        ):

            return "COMPLETE"


        print(
            f"\nInvalid patch found; "
            f"deleting:\n{tif}"
        )


        tif.unlink(
            missing_ok=True
        )


    return "PENDING"


# ============================================================
# 31. WRITE COMPLETED PATCH LOCALLY
# ============================================================

def write_local_patch(
    patch_array,
    spec,
    source,
    scaling_factor
):

    LOCAL_PATCH_FILE.unlink(
        missing_ok=True
    )


    patch_window = (
        spec[
            "window"
        ]
    )


    patch_transform = (
        window_transform(

            patch_window,

            source.transform
        )
    )


    profile = {

        "driver":
            "GTiff",

        "width":
            int(
                patch_window.width
            ),

        "height":
            int(
                patch_window.height
            ),

        "count":
            1,

        "dtype":
            "float32",

        "crs":
            source.crs,

        "transform":
            patch_transform,

        "nodata":
            OUTPUT_NODATA,

        "compress":
            PATCH_COMPRESSION,

        "predictor":
            3,

        "zlevel":
            PATCH_ZLEVEL,

        "tiled":
            True,

        "blockxsize":
            512,

        "blockysize":
            512,

        "BIGTIFF":
            "IF_SAFER",
    }


    with rasterio.Env(
        GDAL_NUM_THREADS="ALL_CPUS"
    ):


        with rasterio.open(

            LOCAL_PATCH_FILE,

            "w",

            **profile

        ) as dst:


            # =================================================
            # ONE WRITE FOR THE ENTIRE LARGE PATCH
            # =================================================

            dst.write(

                patch_array,

                1
            )


            dst.set_band_description(

                1,

                "Calibrated_RF_ensemble_uncertainty"
            )


            dst.update_tags(

                patch_id=(
                    spec[
                        "patch_id"
                    ]
                ),

                target="SOC",

                units="g C kg-1",

                equation="U_RF(x)=k*s_RF(x)",

                scaling_factor_k=float(
                    scaling_factor
                ),

                B_RF_models=B,

                trees_per_RF=(
                    RF_FIXED_PARAMETERS[
                        "n_estimators"
                    ]
                ),
            )


# ============================================================
# 32. COPY COMPLETED PATCH TO DRIVE
# ============================================================

def copy_patch_to_drive(
    spec
):

    patch_id = (
        spec[
            "patch_id"
        ]
    )


    destination = (
        patch_tif_path(
            patch_id
        )
    )


    partial = (
        patch_partial_path(
            patch_id
        )
    )


    partial.unlink(
        missing_ok=True
    )


    destination.unlink(
        missing_ok=True
    )


    # One sequential file copy
    shutil.copyfile(

        LOCAL_PATCH_FILE,

        partial
    )


    # Atomic-ish final rename after full copy
    os.replace(

        partial,

        destination
    )


    LOCAL_PATCH_FILE.unlink(
        missing_ok=True
    )


# ============================================================
# 33. CREATE EMPTY-PATCH MARKER
# ============================================================

def save_empty_marker(
    spec,
    scaling_factor
):

    marker = (
        patch_empty_marker(
            spec[
                "patch_id"
            ]
        )
    )


    information = {

        "patch_id":
            spec[
                "patch_id"
            ],

        "status":
            "EMPTY",

        "scaling_factor_k":
            float(
                scaling_factor
            ),

        "window": {

            "col_off":
                int(
                    spec[
                        "window"
                    ].col_off
                ),

            "row_off":
                int(
                    spec[
                        "window"
                    ].row_off
                ),

            "width":
                int(
                    spec[
                        "window"
                    ].width
                ),

            "height":
                int(
                    spec[
                        "window"
                    ].height
                ),
        }
    }


    with open(
        marker,
        "w"
    ) as file:

        json.dump(

            information,

            file,

            indent=4
        )


# ============================================================
# 34. RUN CONFIGURATION PROTECTION
#
# Prevent accidental reuse of patches generated using a
# different k / feature set / patch size / RF configuration.
# ============================================================

def validate_run_configuration(
    scaling_factor
):

    config = {

        "input_raster":
            str(
                INPUT_STACKED_RASTER
            ),

        "selected_features":
            SELECTED_FEATURES,

        "B":
            B,

        "rf_parameters":
            RF_FIXED_PARAMETERS,

        "patch_size":
            PATCH_SIZE,

        "prediction_window_size":
            PREDICTION_WINDOW_SIZE,

        "scaling_factor_k":
            float(
                scaling_factor
            ),
    }


    if OVERWRITE_PATCHES:

        if PATCH_DIR.exists():

            shutil.rmtree(
                PATCH_DIR
            )


        PATCH_DIR.mkdir(
            parents=True,
            exist_ok=True
        )


        RUN_CONFIG_JSON.unlink(
            missing_ok=True
        )


        PATCH_MANIFEST_CSV.unlink(
            missing_ok=True
        )


    if RUN_CONFIG_JSON.exists():

        with open(
            RUN_CONFIG_JSON,
            "r"
        ) as file:

            previous = json.load(
                file
            )


        structural_checks = [

            previous.get(
                "input_raster"
            )
            == config[
                "input_raster"
            ],

            previous.get(
                "selected_features"
            )
            == config[
                "selected_features"
            ],

            previous.get(
                "B"
            )
            == config[
                "B"
            ],

            previous.get(
                "rf_parameters"
            )
            == config[
                "rf_parameters"
            ],

            previous.get(
                "patch_size"
            )
            == config[
                "patch_size"
            ],

            previous.get(
                "prediction_window_size"
            )
            == config[
                "prediction_window_size"
            ],
        ]


        previous_k = float(

            previous.get(
                "scaling_factor_k"
            )
        )


        if (

            not all(
                structural_checks
            )

            or not np.isclose(
                previous_k,
                scaling_factor
            )

        ):

            raise RuntimeError(

                "\nExisting patches were generated "
                "with a different configuration.\n\n"

                "If you intentionally changed the "
                "method/settings, set:\n\n"

                "OVERWRITE_PATCHES = True"
            )


    else:

        with open(
            RUN_CONFIG_JSON,
            "w"
        ) as file:

            json.dump(

                config,

                file,

                indent=4
            )


# ============================================================
# 35. SAVE PATCH MANIFEST
# ============================================================

def save_manifest(
    manifest_rows
):

    pd.DataFrame(
        manifest_rows
    ).to_csv(

        PATCH_MANIFEST_CSV,

        index=False
    )


# ============================================================
# 36. PROCESS ONE LARGE PATCH
# ============================================================

def process_large_patch(
    source,
    models,
    scaling_factor,
    band_indices,
    spec
):

    patch_window = (
        spec[
            "window"
        ]
    )


    patch_height = int(
        patch_window.height
    )


    patch_width = int(
        patch_window.width
    )


    # ========================================================
    # ONLY one output array for the whole large patch
    #
    # Example:
    #
    # 4096 × 4096 × 4 bytes
    # ≈ 67 MB
    # ========================================================

    patch_array = np.full(

        (
            patch_height,
            patch_width
        ),

        OUTPUT_NODATA,

        dtype=np.float32
    )


    total_valid_pixels = 0


    internal_counter = 0


    first_band = (
        band_indices[0]
    )


    internal_windows = list(

        generate_internal_windows(
            patch_window
        )
    )


    total_internal = len(
        internal_windows
    )


    for (

        national_window,

        local_row,

        local_col,

        sub_height,

        sub_width

    ) in internal_windows:


        internal_counter += 1


        # ----------------------------------------------------
        # Skip obvious NoData pieces
        # ----------------------------------------------------

        if not window_has_possible_data(

            source,

            first_band,

            national_window

        ):

            continue


        data, valid = (
            read_predictor_window(

                source,

                band_indices,

                national_window
            )
        )


        n_valid = int(

            np.count_nonzero(
                valid
            )
        )


        if n_valid > 0:

            X = np.ascontiguousarray(

                data[
                    :,
                    valid
                ].T,

                dtype=np.float32
            )


            raw_spread = (
                predict_spread_in_batches(

                    models,

                    X
                )
            )


            calibrated = (

                scaling_factor
                * raw_spread
            )


            # =================================================
            # View into corresponding part of 4096 patch
            # =================================================

            patch_view = (

                patch_array[

                    local_row:
                    local_row + sub_height,

                    local_col:
                    local_col + sub_width
                ]
            )


            patch_view[
                valid
            ] = calibrated.astype(
                np.float32
            )


            total_valid_pixels += (
                n_valid
            )


            del X
            del raw_spread
            del calibrated
            del patch_view


        del data
        del valid


        print(

            "\r"

            f"    internal chunk "
            f"{internal_counter:02d}/"
            f"{total_internal:02d} "

            f"| patch valid pixels="
            f"{total_valid_pixels:,}",

            end="",

            flush=True
        )


        if (
            internal_counter
            % GC_EVERY_INTERNAL_WINDOWS
            == 0
        ):

            gc.collect()


    print("\n")


    # ========================================================
    # WHOLE LARGE PATCH IS EMPTY
    # ========================================================

    if total_valid_pixels == 0:

        del patch_array

        gc.collect()


        return (
            "EMPTY",
            0
        )


    # ========================================================
    # WRITE WHOLE PATCH ONCE
    # ========================================================

    write_local_patch(

        patch_array,

        spec,

        source,

        scaling_factor
    )


    del patch_array

    gc.collect()


    # ========================================================
    # COPY WHOLE COMPLETED PATCH ONCE TO DRIVE
    # ========================================================

    copy_patch_to_drive(
        spec
    )


    return (

        "COMPLETE",

        total_valid_pixels
    )


# ============================================================
# 37. PROCESS ALL LARGE PATCHES
# ============================================================

def process_all_patches(
    models,
    scaling_factor
):

    banner(
        "PROCESS LARGE UNCERTAINTY PATCHES"
    )


    manifest_rows = []


    with rasterio.open(
        INPUT_STACKED_RASTER
    ) as source:


        if (
            source.driver.upper()
            == "VRT"
        ):

            raise RuntimeError(
                "\nInput must be the physical "
                "national GeoTIFF, not a VRT."
            )


        band_indices = (
            resolve_raster_band_indices(
                source
            )
        )


        specs = (
            build_patch_specs(

                source.width,

                source.height
            )
        )


        total_patches = len(
            specs
        )


        print(
            f"\nNational raster:"
            f"\n  Width:  "
            f"{source.width:,}"
            f"\n  Height: "
            f"{source.height:,}"
        )


        print(
            f"\nLarge patch size: "
            f"{PATCH_SIZE} × "
            f"{PATCH_SIZE}"
        )


        print(
            f"Internal RF window: "
            f"{PREDICTION_WINDOW_SIZE} × "
            f"{PREDICTION_WINDOW_SIZE}"
        )


        print(
            f"\nTotal large patches: "
            f"{total_patches}"
        )


        completed_before = 0


        # ====================================================
        # PROCESS EACH PATCH
        # ====================================================

        for patch_number, spec in enumerate(
            specs,
            start=1
        ):


            patch_id = (
                spec[
                    "patch_id"
                ]
            )


            status = (
                get_patch_status(

                    spec,

                    source
                )
            )


            # =================================================
            # ALREADY COMPLETE
            # =================================================

            if status in [
                "COMPLETE",
                "EMPTY",
            ]:

                completed_before += 1


                print(

                    f"[{patch_number:03d}/"
                    f"{total_patches:03d}] "

                    f"{patch_id} "

                    f"→ {status} "

                    f"(SKIP)"
                )


                manifest_rows.append({

                    "patch_index":
                        patch_number,

                    "patch_id":
                        patch_id,

                    "status":
                        status,

                    "row_off":
                        int(
                            spec[
                                "window"
                            ].row_off
                        ),

                    "col_off":
                        int(
                            spec[
                                "window"
                            ].col_off
                        ),

                    "width":
                        spec[
                            "width"
                        ],

                    "height":
                        spec[
                            "height"
                        ],

                    "valid_pixels":
                        np.nan,

                    "processing_seconds":
                        0,
                })


                continue


            # =================================================
            # NEW / INCOMPLETE PATCH
            # =================================================

            print("\n")


            print(
                "=" * 80
            )


            print(

                f"PATCH "
                f"{patch_number:03d}/"
                f"{total_patches:03d}"

                f" | "
                f"{patch_id}"
            )


            print(
                "=" * 80
            )


            print(

                f"\nNational window:"
                f"\n  col_off = "
                f"{int(spec['window'].col_off):,}"
                f"\n  row_off = "
                f"{int(spec['window'].row_off):,}"
                f"\n  width   = "
                f"{spec['width']:,}"
                f"\n  height  = "
                f"{spec['height']:,}"
            )


            patch_start = (
                time.perf_counter()
            )


            (
                result_status,
                valid_pixels

            ) = process_large_patch(

                source,

                models,

                scaling_factor,

                band_indices,

                spec
            )


            elapsed = (

                time.perf_counter()
                - patch_start
            )


            # =================================================
            # EMPTY MARKER
            # =================================================

            if result_status == "EMPTY":

                save_empty_marker(

                    spec,

                    scaling_factor
                )


            # =================================================
            # VERIFY EXPORTED PATCH
            # =================================================

            elif result_status == "COMPLETE":

                output_patch = (
                    patch_tif_path(
                        patch_id
                    )
                )


                if not patch_is_valid(

                    output_patch,

                    spec[
                        "window"
                    ],

                    source

                ):

                    raise RuntimeError(

                        f"\nPatch export failed:\n"
                        f"{output_patch}"
                    )


            print(
                f"\n✓ Patch status: "
                f"{result_status}"
            )


            print(
                f"✓ Valid pixels: "
                f"{valid_pixels:,}"
            )


            print(
                f"✓ Processing time: "
                f"{elapsed / 60:.2f} min"
            )


            if result_status == "COMPLETE":

                patch_path = (
                    patch_tif_path(
                        patch_id
                    )
                )


                print(
                    f"✓ Patch size: "
                    f"{human_size(patch_path.stat().st_size)}"
                )


            manifest_rows.append({

                "patch_index":
                    patch_number,

                "patch_id":
                    patch_id,

                "status":
                    result_status,

                "row_off":
                    int(
                        spec[
                            "window"
                        ].row_off
                    ),

                "col_off":
                    int(
                        spec[
                            "window"
                        ].col_off
                    ),

                "width":
                    spec[
                        "width"
                    ],

                "height":
                    spec[
                        "height"
                    ],

                "valid_pixels":
                    valid_pixels,

                "processing_seconds":
                    elapsed,
            })


            # Save after EVERY completed large patch.
            save_manifest(
                manifest_rows
            )


    # ========================================================
    # FINAL PATCH CHECK
    # ========================================================

    banner(
        "PATCH PROCESSING COMPLETE"
    )


    print(
        f"\nTotal expected patches: "
        f"{len(specs)}"
    )


    print(
        f"Previously completed/skipped: "
        f"{completed_before}"
    )


    return (
        specs,
        band_indices
    )


# ============================================================
# 38. MOSAIC CHECKPOINT
# ============================================================

def save_mosaic_checkpoint(
    next_patch_index,
    total_patches
):

    information = {

        "next_patch_index":
            int(
                next_patch_index
            ),

        "total_patches":
            int(
                total_patches
            ),

        "patch_size":
            PATCH_SIZE,

        "partial_raster":
            str(
                MOSAIC_PARTIAL_RASTER
            ),

        "final_raster":
            str(
                FINAL_UNCERTAINTY_RASTER
            ),
    }


    temporary = Path(

        str(
            MOSAIC_CHECKPOINT_FILE
        )

        + ".tmp"
    )


    with open(
        temporary,
        "w"
    ) as file:

        json.dump(

            information,

            file,

            indent=4
        )


    os.replace(

        temporary,

        MOSAIC_CHECKPOINT_FILE
    )


def load_mosaic_checkpoint():

    if not MOSAIC_CHECKPOINT_FILE.exists():

        return None


    with open(
        MOSAIC_CHECKPOINT_FILE,
        "r"
    ) as file:

        return json.load(
            file
        )


# ============================================================
# 39. VALIDATE FINAL/PARTIAL NATIONAL RASTER
# ============================================================

def national_raster_is_valid(
    path,
    source
):

    if not Path(
        path
    ).exists():

        return False


    try:

        with rasterio.open(
            path
        ) as raster:

            return all([

                raster.width
                == source.width,

                raster.height
                == source.height,

                raster.count
                == 1,

                raster.crs
                == source.crs,

                raster.transform
                == source.transform,

                raster.dtypes[0]
                == "float32",
            ])


    except Exception:

        return False


# ============================================================
# 40. CREATE EMPTY FINAL MOSAIC
# ============================================================

def create_partial_mosaic(
    source,
    scaling_factor
):

    MOSAIC_PARTIAL_RASTER.unlink(
        missing_ok=True
    )


    profile = {

        "driver":
            "GTiff",

        "width":
            source.width,

        "height":
            source.height,

        "count":
            1,

        "dtype":
            "float32",

        "crs":
            source.crs,

        "transform":
            source.transform,

        "nodata":
            OUTPUT_NODATA,

        "compress":
            MOSAIC_COMPRESSION,

        "predictor":
            3,

        "zlevel":
            MOSAIC_ZLEVEL,

        "tiled":
            True,

        "blockxsize":
            512,

        "blockysize":
            512,

        "BIGTIFF":
            "YES",

        # Empty national areas remain unwritten.
        "SPARSE_OK":
            "YES",
    }


    banner(
        "CREATE NATIONAL MOSAIC"
    )


    with rasterio.Env(
        GDAL_NUM_THREADS="ALL_CPUS"
    ):


        with rasterio.open(

            MOSAIC_PARTIAL_RASTER,

            "w",

            **profile

        ) as dst:


            dst.set_band_description(

                1,

                "Calibrated_RF_ensemble_uncertainty"
            )


            dst.update_tags(

                target="SOC",

                units="g C kg-1",

                equation="U_RF(x)=k*s_RF(x)",

                scaling_factor_k=float(
                    scaling_factor
                ),

                B_RF_models=B,

                trees_per_RF=(
                    RF_FIXED_PARAMETERS[
                        "n_estimators"
                    ]
                ),

                patch_size=PATCH_SIZE,
            )


# ============================================================
# 41. MOSAIC ALL PATCHES
#
# Only ~105 large writes when PATCH_SIZE=4096.
# ============================================================

def mosaic_all_patches(
    specs,
    scaling_factor
):

    banner(
        "MOSAIC LARGE UNCERTAINTY PATCHES"
    )


    with rasterio.open(
        INPUT_STACKED_RASTER
    ) as source:


        # ====================================================
        # Existing final output
        # ====================================================

        if (

            FINAL_UNCERTAINTY_RASTER.exists()

            and not FORCE_REBUILD_MOSAIC

        ):

            if national_raster_is_valid(

                FINAL_UNCERTAINTY_RASTER,

                source

            ):

                print(
                    "\n✓ Final national uncertainty "
                    "raster already exists."
                )


                print(
                    "Skipping mosaic."
                )


                return


        if FORCE_REBUILD_MOSAIC:

            FINAL_UNCERTAINTY_RASTER.unlink(
                missing_ok=True
            )


            MOSAIC_PARTIAL_RASTER.unlink(
                missing_ok=True
            )


            MOSAIC_CHECKPOINT_FILE.unlink(
                missing_ok=True
            )


        # ====================================================
        # Resume mosaic
        # ====================================================

        checkpoint = (
            load_mosaic_checkpoint()
        )


        start_patch = 0


        if (

            checkpoint is not None

            and national_raster_is_valid(

                MOSAIC_PARTIAL_RASTER,

                source
            )

        ):

            start_patch = int(

                checkpoint.get(
                    "next_patch_index",
                    0
                )
            )


            print(
                f"\nResuming mosaic from "
                f"patch "
                f"{start_patch + 1}/"
                f"{len(specs)}"
            )


        else:

            create_partial_mosaic(

                source,

                scaling_factor
            )


            start_patch = 0


        # ====================================================
        # OPEN NATIONAL OUTPUT ONCE
        # ====================================================

        with rasterio.Env(
            GDAL_NUM_THREADS="ALL_CPUS"
        ):


            with rasterio.open(

                MOSAIC_PARTIAL_RASTER,

                "r+"

            ) as destination:


                for index, spec in enumerate(
                    specs
                ):


                    if index < start_patch:

                        continue


                    patch_id = (
                        spec[
                            "patch_id"
                        ]
                    )


                    empty_marker = (
                        patch_empty_marker(
                            patch_id
                        )
                    )


                    patch_path = (
                        patch_tif_path(
                            patch_id
                        )
                    )


                    # =========================================
                    # Empty patch: nothing to write
                    # =========================================

                    if empty_marker.exists():

                        print(

                            f"\rMosaic "
                            f"{index + 1:03d}/"
                            f"{len(specs):03d} "

                            f"{patch_id} "
                            f"| EMPTY",

                            end="",

                            flush=True
                        )


                    # =========================================
                    # Actual uncertainty patch
                    # =========================================

                    else:

                        if not patch_is_valid(

                            patch_path,

                            spec[
                                "window"
                            ],

                            source

                        ):

                            raise RuntimeError(

                                f"\nMissing or invalid patch:\n"
                                f"{patch_path}"
                            )


                        with rasterio.open(
                            patch_path
                        ) as patch:


                            # Read one LARGE patch.
                            patch_data = patch.read(
                                1
                            )


                            # =================================
                            # ONE write to national mosaic
                            # =================================

                            destination.write(

                                patch_data,

                                1,

                                window=(
                                    spec[
                                        "window"
                                    ]
                                )
                            )


                            del patch_data


                        print(

                            f"\rMosaic "
                            f"{index + 1:03d}/"
                            f"{len(specs):03d} "

                            f"{patch_id} "
                            f"| WRITE",

                            end="",

                            flush=True
                        )


                    completed = (
                        index + 1
                    )


                    if (

                        completed
                        % MOSAIC_CHECKPOINT_EVERY
                        == 0

                        or completed
                        == len(specs)

                    ):

                        save_mosaic_checkpoint(

                            next_patch_index=completed,

                            total_patches=len(
                                specs
                            )
                        )


    print("\n")


    # ========================================================
    # VALIDATE PARTIAL MOSAIC
    # ========================================================

    with rasterio.open(
        INPUT_STACKED_RASTER
    ) as source:


        if not national_raster_is_valid(

            MOSAIC_PARTIAL_RASTER,

            source

        ):

            raise RuntimeError(
                "Completed mosaic is invalid."
            )


    # ========================================================
    # Rename partial to final only after success.
    # ========================================================

    FINAL_UNCERTAINTY_RASTER.unlink(
        missing_ok=True
    )


    os.replace(

        MOSAIC_PARTIAL_RASTER,

        FINAL_UNCERTAINTY_RASTER
    )


    MOSAIC_CHECKPOINT_FILE.unlink(
        missing_ok=True
    )


    print(
        "\n✓ Final mosaic completed."
    )


    print(
        f"\nFinal raster:\n"
        f"{FINAL_UNCERTAINTY_RASTER}"
    )


# ============================================================
# 42. FINAL VALIDATION
# ============================================================

def validate_final_output():

    banner(
        "FINAL VALIDATION"
    )


    with rasterio.open(
        INPUT_STACKED_RASTER
    ) as source:


        if not national_raster_is_valid(

            FINAL_UNCERTAINTY_RASTER,

            source

        ):

            raise RuntimeError(
                "Final uncertainty raster is invalid."
            )


    with rasterio.open(
        FINAL_UNCERTAINTY_RASTER
    ) as output:


        print(
            f"\nFinal raster:"
            f"\n{FINAL_UNCERTAINTY_RASTER}"
        )


        print(
            f"\nDriver:      "
            f"{output.driver}"
        )


        print(
            f"Width:       "
            f"{output.width:,}"
        )


        print(
            f"Height:      "
            f"{output.height:,}"
        )


        print(
            f"Bands:       "
            f"{output.count}"
        )


        print(
            f"CRS:         "
            f"{output.crs}"
        )


        print(
            f"Resolution:  "
            f"{output.res}"
        )


        print(
            f"NoData:      "
            f"{output.nodata}"
        )


        print(
            f"Data type:   "
            f"{output.dtypes[0]}"
        )


        print(
            f"File size:   "
            f"{human_size(FINAL_UNCERTAINTY_RASTER.stat().st_size)}"
        )


# ============================================================
# 43. MAIN
# ============================================================

def main():

    banner(
        "PATCH-BASED CALIBRATED RF ENSEMBLE UNCERTAINTY"
    )


    check_inputs()


    # ========================================================
    # LOAD POINT DATA
    # ========================================================

    banner(
        "1. LOAD DEVELOPMENT + VALIDATION DATA"
    )


    development = (
        load_point_dataset(

            DEVELOPMENT_GPKG,

            "Development dataset"
        )
    )


    validation = (
        load_point_dataset(

            VALIDATION_GPKG,

            "Validation dataset"
        )
    )


    print(
        f"\nDevelopment samples: "
        f"{len(development)}"
    )


    print(
        f"Validation samples: "
        f"{len(validation)}"
    )


    # ========================================================
    # DEVELOPMENT MATRIX
    # ========================================================

    X_development = (

        development[
            SELECTED_FEATURES
        ]

        .to_numpy(
            dtype=np.float32
        )
    )


    y_development = (

        development[
            TARGET_COLUMN
        ]

        .to_numpy(
            dtype=np.float64
        )
    )


    print(
        f"\nDevelopment matrix: "
        f"{X_development.shape}"
    )


    # ========================================================
    # LOAD / TRAIN 50 RF MODELS
    # ========================================================

    banner(
        "2. PREPARE RF ENSEMBLE"
    )


    models = (
        train_or_load_ensemble(

            X_development,

            y_development
        )
    )


    print(
        f"\nModels: "
        f"{len(models)}"
    )


    print(
        f"Trees/model: "
        f"{RF_FIXED_PARAMETERS['n_estimators']:,}"
    )


    print(
        f"Total tree evaluations/pixel: "
        f"{B * RF_FIXED_PARAMETERS['n_estimators']:,}"
    )


    # ========================================================
    # CALIBRATION FACTOR
    # ========================================================

    banner(
        "3. CALIBRATE ENSEMBLE UNCERTAINTY"
    )


    scaling_factor = (
        derive_scaling_factor(

            models,

            validation
        )
    )


    print(
        f"\nFinal calibration factor:"
        f"\nk = {scaling_factor:.8f}"
    )


    # ========================================================
    # CHECK RUN CONFIGURATION
    # ========================================================

    validate_run_configuration(
        scaling_factor
    )


    # ========================================================
    # LARGE PATCH CALCULATION
    # ========================================================

    banner(
        "4. CALCULATE LARGE UNCERTAINTY PATCHES"
    )


    (
        patch_specs,
        band_indices

    ) = process_all_patches(

        models,

        scaling_factor
    )


    # ========================================================
    # NATIONAL MOSAIC
    # ========================================================

    banner(
        "5. MOSAIC LARGE PATCHES"
    )


    mosaic_all_patches(

        patch_specs,

        scaling_factor
    )


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    validate_final_output()


    # ========================================================
    # COMPLETE
    # ========================================================

    banner(
        "PROCESS COMPLETED SUCCESSFULLY"
    )


    print(
        f"\nLarge patch size:"
        f"\n{PATCH_SIZE} × {PATCH_SIZE}"
    )


    print(
        f"\nInternal prediction window:"
        f"\n{PREDICTION_WINDOW_SIZE} × "
        f"{PREDICTION_WINDOW_SIZE}"
    )


    print(
        f"\nNumber of predictors:"
        f"\n{N_FEATURES}"
    )


    print(
        f"\nB:"
        f"\n{B}"
    )


    print(
        f"\nTrees per RF:"
        f"\n{RF_FIXED_PARAMETERS['n_estimators']}"
    )


    print(
        f"\nCalibration factor:"
        f"\nk = {scaling_factor:.8f}"
    )


    print(
        "\nLarge uncertainty patches:"
    )


    print(
        PATCH_DIR
    )


    print(
        "\nPatch manifest:"
    )


    print(
        PATCH_MANIFEST_CSV
    )


    print(
        "\nFinal national uncertainty raster:"
    )


    print(
        FINAL_UNCERTAINTY_RASTER
    )


    print(
        "\nCalibration summary:"
    )


    print(
        CALIBRATION_SUMMARY_CSV
    )


# ============================================================
# 44. RUN
# ============================================================

if __name__ == "__main__":

    main()


PATCH-BASED CALIBRATED RF ENSEMBLE UNCERTAINTY

1. LOAD DEVELOPMENT + VALIDATION DATA
[16:33:36] Loading Development dataset
[16:33:37] Loading Validation dataset

Development samples: 385
Validation samples: 43

Development matrix: (385, 33)

2. PREPARE RF ENSEMBLE

TRAIN / LOAD B=50 RF MODELS
[16:33:37] Loaded RF 01/50
[16:33:38] Loaded RF 02/50
[16:33:38] Loaded RF 03/50
[16:33:39] Loaded RF 04/50
[16:33:39] Loaded RF 05/50
[16:33:40] Loaded RF 06/50
[16:33:41] Loaded RF 07/50
[16:33:41] Loaded RF 08/50
[16:33:42] Loaded RF 09/50
[16:33:43] Loaded RF 10/50
[16:33:43] Loaded RF 11/50
[16:33:44] Loaded RF 12/50
[16:33:44] Loaded RF 13/50
[16:33:45] Loaded RF 14/50
[16:33:45] Loaded RF 15/50
[16:33:46] Loaded RF 16/50
[16:33:46] Loaded RF 17/50
[16:33:47] Loaded RF 18/50
[16:33:48] Loaded RF 19/50
[16:33:48] Loaded RF 20/50
[16:33:49] Loaded RF 21/50
[16:33:49] Loaded RF 22/50
[16:33:50] Loaded RF 23/50
[16:33:50] Loaded RF 24/50
[16:33:51] Loaded RF 25/50
[16:33:51] Loaded RF 26/50
[